In [ ]:
import os

for root, dirs, files in os.walk('/content'):
    print(root)
    print(files)
    print('-'*50)

In [ ]:
!find /content -name "*.jsonl"

In [ ]:
!ls /content

In [ ]:
import os

print(os.getcwd())
print(os.listdir("."))

In [ ]:
with open('/content/candidates.jsonl', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        try:
            json.loads(line)
        except Exception as e:
            print(f"Error at line {i+1}")
            print(line[:500])  # first 500 chars
            print(e)
            break

In [ ]:
with open('/content/candidates.jsonl', 'r', encoding='utf-8') as f:
    for _ in range(5):
        print(next(f))

In [ ]:
import json

candidates = []

with open('/content/candidates.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        candidates.append(json.loads(line))

print("Total Candidates:", len(candidates))

In [ ]:
print(candidates[0].keys())

In [ ]:
import pprint

pprint.pprint(candidates[0])

In [ ]:
print(json.dumps(candidates[0], indent=4))

In [ ]:
print("Total Candidates:", len(candidates))
print(candidates[0].keys())
pprint.pprint(candidates[0])

In [ ]:
!pip install python-docx

In [ ]:
from docx import Document

doc = Document('/content/job_description.docx')

jd = ""

for para in doc.paragraphs:
    jd += para.text + "\n"

print(jd)

In [ ]:
def create_candidate_text(candidate):

    text = ""

    # Profile
    text += candidate['profile']['headline'] + " "
    text += candidate['profile']['summary'] + " "

    # Skills
    for skill in candidate['skills']:
        text += skill['name'] + " "

    # Career history
    for job in candidate['career_history']:
        text += job['title'] + " "
        text += job['industry'] + " "
        text += job['description'] + " "

    return text

In [ ]:
print(create_candidate_text(candidates[0])[:1000])

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

In [ ]:
jd_embedding = model.encode(jd)

In [ ]:
subset = candidates

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

print("Creating candidate texts...")

candidate_texts = []

for candidate in subset:
    candidate_texts.append(
        create_candidate_text(candidate)
    )

print("Generating embeddings in batches...")

candidate_embeddings = model.encode(
    candidate_texts,
    batch_size=256,
    show_progress_bar=True
)

print("Generating JD embedding...")

jd_embedding = model.encode(jd)

print("Calculating similarities...")

similarities = cosine_similarity(
    [jd_embedding],
    candidate_embeddings
)[0]

print("Creating scores...")

scores = []

for idx, candidate in enumerate(subset):

    scores.append(
        (
            candidate["candidate_id"],
            similarities[idx]
        )
    )

scores = sorted(
    scores,
    key=lambda x: x[1],
    reverse=True
)

print(scores[:10])

In [ ]:
service_companies = [
    "TCS",
    "Infosys",
    "Wipro",
    "Accenture",
    "Cognizant",
    "Capgemini"
]

In [ ]:
def company_score(candidate):

    score = 0

    for job in candidate['career_history']:

        if job['company'] not in service_companies:
            score += 1

    return min(score/3,1)

In [ ]:
production_keywords = [

    "retrieval",
    "ranking",
    "recommendation",
    "embeddings",
    "production",
    "deployed",
    "pipeline",
    "evaluation",
    "A/B",
    "hybrid search",
    "vector",
    "real-time",
    "search",
    "index",
    "ML models"
]

In [ ]:
def production_score(candidate):

    text = create_candidate_text(candidate).lower()

    score = 0

    for word in production_keywords:

        if word.lower() in text:
            score += 1

    return score/len(production_keywords)

In [ ]:
def behavior_score(candidate):

    s = candidate['redrob_signals']

    score = 0

    if s['open_to_work_flag']:
        score += 0.25

    score += min(
        s['recruiter_response_rate'],
        1
    ) * 0.25

    score += min(
        s['interview_completion_rate'],
        1
    ) * 0.25

    score += (
        s['github_activity_score']/10
    ) * 0.25

    return score

In [ ]:
def experience_score(candidate):

    exp = candidate['profile']['years_of_experience']

    if 5 <= exp <= 9:
        return 1

    elif exp > 9:
        return 0.8

    else:
        return exp/5

In [ ]:
def behavior_score(candidate):

    s = candidate['redrob_signals']

    open_to_work = 1 if s['open_to_work_flag'] else 0

    recruiter_response = min(
        s['recruiter_response_rate'], 1
    )

    interview_completion = min(
        s['interview_completion_rate'], 1
    )

    github_score = min(
        s['github_activity_score'] / 10, 1
    )

    score = (
        0.25 * open_to_work +
        0.25 * recruiter_response +
        0.25 * interview_completion +
        0.25 * github_score
    )

    return min(score, 1)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

results = []

subset = candidates[:100]   # Test on first 100 candidates

for candidate in subset:

    # Semantic Similarity
    candidate_text = create_candidate_text(candidate)

    candidate_embedding = model.encode(candidate_text)

    semantic_similarity = cosine_similarity(
        [jd_embedding],
        [candidate_embedding]
    )[0][0]

    # Other scores
    prod_score = production_score(candidate)
    beh_score = behavior_score(candidate)
    comp_score = company_score(candidate)
    exp_score = experience_score(candidate)

    # Final score
    final_score = (
        0.45 * semantic_similarity +
        0.20 * prod_score +
        0.15 * beh_score +
        0.10 * comp_score +
        0.10 * exp_score
    )

    results.append({
        "candidate_id": candidate["candidate_id"],
        "semantic_score": semantic_similarity,
        "production_score": prod_score,
        "behavior_score": beh_score,
        "company_score": comp_score,
        "experience_score": exp_score,
        "final_score": final_score
    })

In [ ]:
results = sorted(
    results,
    key=lambda x: x["final_score"],
    reverse=True
)

In [ ]:
for r in results[:10]:
    print(r)

In [ ]:
s = candidate['redrob_signals']

print("Open to work:", s['open_to_work_flag'])
print("Recruiter response:", s['recruiter_response_rate'])
print("Interview completion:", s['interview_completion_rate'])
print("Github:", s['github_activity_score'])

In [ ]:
for i in range(5):
    print(
        candidates[i]['candidate_id'],
        behavior_score(candidates[i])
    )

In [ ]:
print(behavior_score(candidates[0]))
print(behavior_score(candidates[1]))
print(behavior_score(candidates[2]))

In [ ]:
import pandas as pd

df = pd.DataFrame(results)

df = df.sort_values(
    by="final_score",
    ascending=False
)

df.to_csv(
    "/content/hybrid_rankings.csv",
    index=False
)

df.head(10)

In [ ]:
top_50 = df.head(50)

top_50

In [ ]:
def explain_candidate(candidate):

    reasons = []

    if production_score(candidate) > 0.3:
        reasons.append(
            "Strong production ML experience"
        )

    if experience_score(candidate) == 1:
        reasons.append(
            "Experience aligns with role"
        )

    if behavior_score(candidate) > 0.7:
        reasons.append(
            "Highly active and responsive candidate"
        )

    if company_score(candidate) > 0.7:
        reasons.append(
            "Strong product company background"
        )

    skills = []

    for s in candidate["skills"][:5]:
        skills.append(s["name"])

    reasons.append(
        f"Key skills: {', '.join(skills)}"
    )

    return reasons

In [ ]:
for r in results[:5]:

    candidate_id = r["candidate_id"]

    candidate = next(
        c for c in candidates
        if c["candidate_id"] == candidate_id
    )

    print("=" * 50)

    print("Candidate:", candidate_id)

    print(
        "Final Score:",
        round(r["final_score"], 3)
    )

    explanations = explain_candidate(
        candidate
    )

    for e in explanations:
        print("✓", e)

    print()

In [ ]:
from docx import Document

doc = Document(
    "/content/redrob_signals_doc.docx"
)

text = ""

for para in doc.paragraphs:
    text += para.text + "\n"

print(text)

In [ ]:
def calculate_final_score(candidate,
                          semantic_score):

    prod = production_score(candidate)

    comp = company_score(candidate)

    exp = experience_score(candidate)

    behavior = behavior_score(candidate)

    base_score = (

          0.60 * semantic_score
        + 0.20 * prod
        + 0.10 * comp
        + 0.10 * exp

    )

    # Behavior acts as multiplier

    multiplier = 0.8 + (0.2 * behavior)

    return base_score * multiplier

In [ ]:
model

In [ ]:
candidate_texts = []

for candidate in subset:
    candidate_texts.append(
        create_candidate_text(candidate)
    )

print(len(candidate_texts))

In [ ]:
candidate_embeddings = model.encode(
    candidate_texts,
    batch_size=32,
    show_progress_bar=True
)

print(candidate_embeddings.shape)

In [ ]:
jd_embedding = model.encode(jd)

print(jd_embedding.shape)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(
    [jd_embedding],
    candidate_embeddings
)[0]

print(similarities[:5])

In [ ]:
results = []

for idx, candidate in enumerate(subset):

    semantic_similarity = similarities[idx]

    final = calculate_final_score(
        candidate,
        semantic_similarity
    )

    results.append({

        "candidate_id":
            candidate["candidate_id"],

        "semantic_score":
            semantic_similarity,

        "behavior_score":
            behavior_score(candidate),

        "final_score":
            final
    })

results = sorted(
    results,
    key=lambda x: x["final_score"],
    reverse=True
)

for r in results[:10]:
    print(r)

In [ ]:
!pip install groq

In [ ]:
from groq import Groq

client = Groq(
    api_key="Your_groq_api_key"
)

In [ ]:
response = client.chat.completions.create(

    model="llama-3.3-70b-versatile",

    messages=[
        {
            "role": "user",
            "content": "Explain RAG in one sentence."
        }
    ]
)

print(response.choices[0].message.content)

In [ ]:
def recruiter_rerank(candidate):

    prompt = f"""
You are a Senior Recruiter hiring for a Founding Senior AI Engineer.

IMPORTANT:

Strongly prefer candidates with:
- Retrieval Systems
- Search Systems
- Ranking Systems
- Recommendation Systems
- NLP
- Embeddings
- Vector Databases
- Production ML Systems
- Product-company experience

Reject candidates whose primary background is:
- Marketing
- Graphic Design
- Sales
- Customer Support
- Pure Computer Vision
- Pure Speech AI
- Non-technical roles

Job Description Summary:

Senior AI Engineer focused on Retrieval, Ranking,
Embeddings, Vector Databases and Production ML.

Candidate Information:

Current Title:
{candidate['profile']['current_title']}

Years of Experience:
{candidate['profile']['years_of_experience']}

Headline:
{candidate['profile']['headline']}

Top Skills:
{[skill['name'] for skill in candidate['skills'][:10]]}

Career Titles:
{[job['title'] for job in candidate['career_history']]}

Behavioral Signals:

Open To Work:
{candidate['redrob_signals']['open_to_work_flag']}

Recruiter Response Rate:
{candidate['redrob_signals']['recruiter_response_rate']}

Profile Completeness:
{candidate['redrob_signals']['profile_completeness_score']}

Notice Period:
{candidate['redrob_signals']['notice_period_days']}

Return ONLY in this format:

FIT_SCORE: <0-100>

STRENGTHS:
- point 1
- point 2

RISKS:
- point 1
- point 2

FINAL_DECISION:
Strong Hire / Hire / Maybe / Reject
"""

    response = client.chat.completions.create(

        # CHANGED MODEL
        model="llama-3.1-8b-instant",

        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],

        temperature=0.1
    )

    return response.choices[0].message.content

In [ ]:
candidate = next(
    c for c in subset
    if c["candidate_id"] == "CAND_0000031"
)

result = recruiter_rerank(candidate)

print(result)

In [ ]:
llm_results = []

top_20 = results[:20]

for r in top_20:

    candidate = next(
        c for c in subset
        if c["candidate_id"] == r["candidate_id"]
    )

    print("=" * 100)
    print("Evaluating:",
          candidate["candidate_id"])

    llm_output = recruiter_rerank(
        candidate
    )

    print(llm_output)

    llm_results.append({

        "candidate_id":
            candidate["candidate_id"],

        "hybrid_score":
            r["final_score"],

        "llm_output":
            llm_output
    })

In [ ]:
import re

def extract_fit_score(text):

    match = re.search(
        r"FIT_SCORE:\s*(\d+)",
        text
    )

    if match:
        return int(match.group(1))

    return 0

In [ ]:
for item in llm_results:

    if item["llm_output"] is None:
        print(
            "Missing output for:",
            item["candidate_id"]
        )

In [ ]:
candidate = next(
    c for c in subset
    if c["candidate_id"] == "CAND_0000031"
)

output = recruiter_rerank(candidate)

print(output)
print(type(output))

In [ ]:
final_results = []

for item in llm_results:

    llm_score = extract_fit_score(
        item["llm_output"]
    )

    final_score = (

         0.4 * item["hybrid_score"] +
         0.6 * (llm_score / 100)

    )

    final_results.append({

        "candidate_id":
            item["candidate_id"],

        "hybrid_score":
            item["hybrid_score"],

        "llm_score":
            llm_score,

        "final_score":
            final_score
    })

final_results = sorted(

    final_results,

    key=lambda x: x["final_score"],

    reverse=True

)

In [ ]:
for rank, r in enumerate(
        final_results, 1):

    print(
        f"{rank}. "
        f"{r['candidate_id']} | "
        f"Hybrid={r['hybrid_score']:.3f} | "
        f"LLM={r['llm_score']} | "
        f"Final={r['final_score']:.3f}"
    )

In [ ]:
import pandas as pd

submission = []

for rank, r in enumerate(
        final_results, 1):

    submission.append({

        "rank": rank,

        "candidate_id":
            r["candidate_id"],

        "score":
            r["final_score"]

    })

submission_df = pd.DataFrame(
    submission
)

submission_df.to_csv(

    "/content/final_submission.csv",

    index=False

)

submission_df.head()

In [ ]:
explanations = []

for item in llm_results:

    explanations.append({

        "candidate_id":
            item["candidate_id"],

        "llm_explanation":
            item["llm_output"]

    })

pd.DataFrame(explanations).to_csv(

    "/content/explanations.csv",

    index=False

)

In [ ]:
!ls /content

In [ ]:
from google.colab import files

files.download('/content/final_submission.csv')

In [ ]:
files.download('/content/explanations.csv')